In [2]:
import pandas as pd
df = pd.DataFrame({
'StudentID': [1, 2, 3, 4, 5, 6],
'Height_cm': [165, 172, 158, 180, 175, 168],
'Weight_kg': [60, 85, 52, 95, 70, 65],
'Score_Midterm': [78, 85, 91, 65, 72, 88],
'Score_Final': [82, 80, 95, 70, 68, 90],
'EnrollDate': pd.to_datetime(['2023-06-01','2022-09-15',
'2023-01-10','2021-03-22',
'2022-07-04','2023-08-19']),
})
print(df.dtypes)


StudentID                 int64
Height_cm                 int64
Weight_kg                 int64
Score_Midterm             int64
Score_Final               int64
EnrollDate       datetime64[ns]
dtype: object


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

df['Score_Improvement'] = df['Score_Final'] - df['Score_Midterm']
df['Score_Total'] = df['Score_Midterm'] + df['Score_Final']

df['BMI'] = df['Weight_kg'] / ((df['Height_cm'] / 100) ** 2)
df['BMI'] = df['BMI'].round(2)
df['Score_Ratio'] = (df['Score_Final'] / df['Score_Midterm']).round(3)

print(df[['Height_cm','Weight_kg','BMI','Score_Midterm',
'Score_Final','Score_Improvement','Score_Ratio']])

poly=PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_num=df[['Score_Midterm', 'Score_Final']]
X_poly=poly.fit_transform(X_num)
poly_cols = poly.get_feature_names_out(['Score_Midterm', 'Score_Final'])
df_poly = pd.DataFrame(X_poly, columns=poly_cols)
print(df_poly.head(3))

   Height_cm  Weight_kg    BMI  Score_Midterm  Score_Final  Score_Improvement  \
0        165         60  22.04             78           82                  4   
1        172         85  28.73             85           80                 -5   
2        158         52  20.83             91           95                  4   
3        180         95  29.32             65           70                  5   
4        175         70  22.86             72           68                 -4   
5        168         65  23.03             88           90                  2   

   Score_Ratio  
0        1.051  
1        0.941  
2        1.044  
3        1.077  
4        0.944  
5        1.023  


In [8]:
import pandas as pd
import numpy as np

df['EnrollDate'] = pd.to_datetime(df['EnrollDate'])
df['Enroll_Year'] = df['EnrollDate'].dt.year
df['Enroll_Month'] = df['EnrollDate'].dt.month
df['Enroll_Day'] = df['EnrollDate'].dt.day
df['Enroll_DayOfWeek'] = df['EnrollDate'].dt.dayofweek
df['Enroll_Quarter'] = df['EnrollDate'].dt.quarter
df['Enroll_IsWeekend'] = df['EnrollDate'].dt.dayofweek.isin([5, 6]).astype(int)
print(df[['EnrollDate','Enroll_Year','Enroll_Month',
'Enroll_DayOfWeek','Enroll_IsWeekend']].head(3))

reference_date = pd.Timestamp('2024-01-01')
df['Days_Since_Enroll'] = (reference_date - df['EnrollDate']).dt.days
print(df[['EnrollDate', 'Days_Since_Enroll']].head(3))

df['Month_sin'] = np.sin(2 * np.pi * df['Enroll_Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Enroll_Month'] / 12)
print(df[['Enroll_Month','Month_sin','Month_cos']].head(4).round(3))

  EnrollDate  Enroll_Year  Enroll_Month  Enroll_DayOfWeek  Enroll_IsWeekend
0 2023-06-01         2023             6                 3                 0
1 2022-09-15         2022             9                 3                 0
2 2023-01-10         2023             1                 1                 0
  EnrollDate  Days_Since_Enroll
0 2023-06-01                214
1 2022-09-15                473
2 2023-01-10                356
   Enroll_Month  Month_sin  Month_cos
0             6        0.0     -1.000
1             9       -1.0     -0.000
2             1        0.5      0.866
3             3        1.0      0.000


In [ ]:
# curse of dimensionality---Irrelevant features add noise, slow down training,
# increase memory usage, and can actually reduce model accuracy by diluting the signal from informative features.

# There are three broad families of selection methods. Filter methods score each feature independently
# using a statistical test (correlation, chi-squared, mutual information) without involving a model — they are fast and
# model-agnostic. Wrapper methods (such as RFE) iteratively train a model and use its performance to decide which
# features to keep — they are more accurate but computationally expensive. Embedded methods perform selection
# as part of model training itself (LASSO regularisation, tree-based feature importances) — they strike a balance
# between speed and accuracy

In [ ]:
| Aspect                                  | Variance Threshold                     | Correlation Filter                        | SelectKBest                                     | RFE                                         | Embedded Methods (LASSO, Trees)        |
| --------------------------------------- | -------------------------------------- | ----------------------------------------- | ----------------------------------------------- | ------------------------------------------- | -------------------------------------- |
| **Category**                            | Filter                                 | Filter                                    | Filter                                          | Wrapper                                     | Embedded                               |
| **Uses Target Variable?**               | ❌ No                                  | ❌ No                                     | ✅ Yes                                         | ✅ Yes                                     | ✅ Yes                                  |
| **Trains a Model?**                     | ❌ No                                  | ❌ No                                     | ❌ No                                          | ✅ Yes                                     | ✅ During training                      |
| **Idea**                                | Remove low-variance features           | Remove highly correlated features         | Keep top K features based on statistical scores | Recursively remove least important features | Model performs selection automatically |
| **Evaluates Features Individually?**    | Yes                                    | Yes (pairwise redundancy)                 | Yes                                             | No                                          | No                                     |
| **Considers Feature Interactions?**     | ❌ No                                  | ❌ No                                     | ❌ No                                          | ✅ Partially                               | ✅ Depends on model                     |
| **Computational Cost**                  | Very Low                               | Low                                       | Low                                             | High                                        | Medium                                 |
| **Speed**                               | Very Fast                              | Fast                                      | Fast                                            | Slow                                        | Moderate                               |
| **Suitable for Large Datasets?**        | ✅ Excellent                           | ✅ Excellent                              | ✅ Good                                        | ❌ Expensive                               | ✅ Good                                 |
| **Requires Scaling?**                   | No                                     | No                                        | Depends on score function                       | Depends on estimator                        | Depends on model                       |
| **Can Detect Nonlinear Relationships?** | ❌ No                                  | ❌ No                                     | Mutual Information only                        | Depends on estimator                        | Tree models can                        |
| **Main Purpose**                        | Remove constant/near-constant columns  | Remove redundant columns                  | Select individually strong predictors           | Find best feature subset                    | Simultaneously train and select        |
| **Example Methods**                     | `VarianceThreshold()`                  | Correlation matrix                        | `SelectKBest()`                                 | `RFE()`, `RFECV()`                          | LASSO, Random Forest, XGBoost          |
| **Output**                              | Removes low-variance columns           | Removes one feature from correlated pairs | Top K features                                  | Best N features                             | Features with non-zero importance      |
| **Risk**                                | May remove rare but important features | May remove useful correlated features     | Misses interactions between features            | Sensitive to base model                     | Depends heavily on model               |
| **Best Use Case**                       | First cleaning step                    | Remove multicollinearity                  | Quick supervised selection                      | When accuracy matters more than speed       | Final modeling stage                   |


In [12]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

df_sel = pd.DataFrame({
'Score_Midterm': [78, 85, 91, 65, 72, 88],
'Score_Final': [82, 80, 95, 70, 68, 90],
'Score_Total': [160,165,186,135,140,178], # = Midterm + Final
'BMI': [22.0,28.8,20.8,29.3,22.9,23.0],
'Constant_Col': [1, 1, 1, 1, 1, 1], # zero variance
'Near_Const': [0, 0, 0, 0, 0, 1], # very low variance
'Target': [1, 0, 1, 0, 1, 1],
})

#variance
features = df_sel.drop(columns=['Target'])
selector = VarianceThreshold(threshold=0.01)
selector.fit(features)
kept = features.columns[selector.get_support()].tolist()
dropped = features.columns[~selector.get_support()].tolist()
print('Kept: ', kept)
print('Dropped:', dropped)
features_vt = selector.transform(features)

#correltion mtrix
# they contain identical information.
# Keeping both:
# increases dimensionality,
# introduces redundancy,
# may create multicollinearity.
df_kept = pd.DataFrame(features_vt, columns=kept)
corr_matrix = df_kept.corr().abs() 
print(corr_matrix.round(3))
threshold = 0.90
upper_tri = corr_matrix.where(
np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
#k=1 ignore the diagonal
#triu means upper triangular.
to_drop = [col for col in upper_tri.columns
if any(upper_tri[col] > threshold)]
print('Dropping due to high correlation:', to_drop)

df_final = df_kept.drop(columns=to_drop)
print(df_final.head())

Kept:  ['Score_Midterm', 'Score_Final', 'Score_Total', 'BMI', 'Near_Const']
Dropped: ['Constant_Col']
               Score_Midterm  Score_Final  Score_Total    BMI  Near_Const
Score_Midterm          1.000        0.912        0.976  0.468       0.399
Score_Final            0.912        1.000        0.979  0.536       0.421
Score_Total            0.976        0.979        1.000  0.515       0.420
BMI                    0.468        0.536        0.515  1.000       0.197
Near_Const             0.399        0.421        0.420  0.197       1.000
Dropping due to high correlation: ['Score_Final', 'Score_Total']
   Score_Midterm   BMI  Near_Const
0           78.0  22.0         0.0
1           85.0  28.8         0.0
2           91.0  20.8         0.0
3           65.0  29.3         0.0
4           72.0  22.9         0.0


In [ ]:
# SelectKBest is a supervised filter method: it scores every feature against the target variable using a statistical test,
# then keeps only the top-K scoring features. The choice of test depends on the data type: f_regression and
# mutual_info_regression are used for continuous targets (regression tasks); f_classif and chi2 are used for
# categorical targets (classification tasks). chi2 requires non-negative features, so it is typically applied after
# MinMaxScaler


# Recursive Feature Elimination (RFE) works differently: it trains a model, inspects the model’s internal feature
# importances (coefficients for linear models, Gini importances for trees), discards the least important feature, retrains,
# and repeats until the desired number of features remains. Because it uses a real model in the loop, RFE is more
# computationally expensive than filter methods but often selects a better feature subset. RFECV additionally performs
# cross-validation at each step to pick the optimal number of features automatically

In [18]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df_ml = pd.DataFrame({
'Score_Midterm': [78,85,91,65,72,88,74,90,60,83],
'Score_Final': [82,80,95,70,68,90,76,92,58,85],
'BMI': [22,29,21,29,23,23,27,20,30,24],
'Days_Since_Enroll':[215,473,356,1016,544,135,290,180,600,400],
'Month_sin': [1.0,0.87,0.5,0.87,0.5,1.0,0.0,-0.5,0.87,0.5],
'Month_cos': [0.0,-0.5,0.87,-0.5,0.87,0.0,-1.0,0.87,-0.5,0.87],
'Score_Improvement':[ 4,-5, 4, 5,-4, 2, 2, 2,-2, 2],
'Score_Ratio': [1.05,0.94,1.04,1.08,0.94,1.02,1.03,1.02,0.97,1.02],
'Target': [1, 0, 1, 0, 1, 1, 0, 1, 0, 1],
})

#Select kBest
X = df_ml.drop(columns=['Target'])
y = df_ml['Target']
selector_k = SelectKBest(score_func=f_classif, k=4)
#k=4----Keep the top 4 highest-scoring features.
selector_k.fit(X, y)
scores = pd.Series(selector_k.scores_, index=X.columns).round(3)
print('F-scores per feature:')
print(scores.sort_values(ascending=False))
#Higher score means stronger relationship with target.
selected_kbest = X.columns[selector_k.get_support()].tolist()
print('Selected by SelectKBest:', selected_kbest)

print("\n\n")
#Recursive Feature Elimination
#Because Logistic Regression coefficients depend on feature scales.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = LogisticRegression(max_iter=500)
rfe = RFE(estimator=model, n_features_to_select=4, step=1)
rfe.fit(X_scaled, y)
rankings = pd.Series(rfe.ranking_, index=X.columns).sort_values()
print('RFE feature rankings (1 = selected):')
print(rankings)
selected_rfe = X.columns[rfe.support_].tolist()
print('Selected by RFE:', selected_rfe)


# SelectKBest
# Examines each feature separately.
# independently.
# Fast.

# RFE
# Evaluates groups of features together.
# Slow but smarter.


# | Rank | Meaning              |
# | ---- | -------------------- |
# | 1    | Selected             |
# | 2    | Removed last         |
# | 3    | Removed earlier      |
# | 4    | Removed even earlier |
# | 5    | Least important      |


F-scores per feature:
BMI                  53.399
Month_cos            23.296
Score_Final           5.278
Score_Midterm         4.788
Days_Since_Enroll     3.965
Score_Improvement     0.526
Month_sin             0.216
Score_Ratio           0.100
dtype: float64
Selected by SelectKBest: ['Score_Midterm', 'Score_Final', 'BMI', 'Month_cos']



RFE feature rankings (1 = selected):
Score_Final          1
BMI                  1
Days_Since_Enroll    1
Month_cos            1
Score_Midterm        2
Score_Improvement    3
Month_sin            4
Score_Ratio          5
dtype: int64
Selected by RFE: ['Score_Final', 'BMI', 'Days_Since_Enroll', 'Month_cos']


In [15]:
# # What is f_classif?
# # Then Final score separates the classes well.
# # Hence it gets a high F-score.

# Choice of score functions
# Classification
# -f_classif--->ANOVA F-test
# -chi2,Chi-square---Requires non-negative values.--->Hence often used after: MinMaxScaler()


# Regression
# -f_regression----ANOVA for continuous targets.
# -mutual_info_regression---Captures nonlinear relationships.